# L07-03｜Qwen3-0.6B LoRA 训练与 loss 分析


## 实验目标

本节完成一轮 50-step 的 LoRA SFT，观察适配器参数更新、训练日志和 loss 曲线。数据集包含 6 条教学样本，重复 9 次后得到 54 条记录，让一个 epoch 足以覆盖 50 个 step。重复样本是为了演示训练过程，不代表增加了新的任务数据。

开始运行前，先预测一下曲线：raw loss 会不会每一步都下降？5-step moving average 会呈现什么趋势？


## 1. 初始化环境、模型和数据

代码会准备 `modelscope`、`ms-swift` 和绘图库，检查 Ascend NPU，下载 `Qwen/Qwen3-0.6B`，并生成 54 条 JSONL 对话数据。

L07-01 中的公式在这里变成训练命令：`W' = W + (alpha / r) × B × A`。基础模型 `W` 保持冻结，优化器更新低秩矩阵 `A`、`B`。


In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import shlex
import shutil
import site
import subprocess
import sys
from pathlib import Path

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

require((3, 10) <= sys.version_info[:2] <= (3, 12), '本流程需要 Python 3.10、3.11 或 3.12；请使用带匹配 Ascend 运行时的 ModelArts 镜像。')
def install_missing_packages() -> None:
    packages = {'modelscope': 'modelscope', 'swift': 'ms-swift', 'matplotlib': 'matplotlib'}
    missing = [dist for module, dist in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        print('安装缺失依赖：', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', *missing])
    python_bin = str(Path(sys.executable).parent)
    user_bin = str(Path(site.getuserbase()) / 'bin')
    old_path = os.environ.get('PATH', '')
    path_entries = old_path.split(os.pathsep) if old_path else []
    for candidate in (python_bin, user_bin):
        if candidate not in path_entries:
            old_path = candidate + os.pathsep + old_path
            path_entries.insert(0, candidate)
    os.environ['PATH'] = old_path

install_missing_packages()

def load_ascend_env() -> None:
    candidates = [
        Path('/usr/local/Ascend/ascend-toolkit/set_env.sh'),
        Path('/usr/local/Ascend/ascend-toolkit/latest/set_env.sh'),
    ]
    ascend_root = Path('/usr/local/Ascend')
    if ascend_root.is_dir():
        candidates.extend(sorted(ascend_root.glob('**/set_env.sh')))
    seen = set()
    for script in candidates:
        if not script.is_file() or str(script) in seen:
            continue
        seen.add(str(script))
        result = subprocess.run(
            ['bash', '-lc', f'source {shlex.quote(str(script))} >/dev/null 2>&1 && env -0'],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
        )
        if result.returncode != 0:
            continue
        for item in result.stdout.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode(errors='ignore')
        print('已加载 Ascend 环境：', script)
        return
    print('未找到 CANN set_env.sh；继续使用当前 ModelArts 进程环境。')

load_ascend_env()
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
import torch
try:
    import torch_npu  # noqa: F401
except Exception as exc:
    raise RuntimeError('当前环境无法导入 torch_npu。请使用带 Ascend/CANN 运行时的 ModelArts 镜像。') from exc
require(hasattr(torch, 'npu'), '当前 PyTorch 没有 torch.npu；请检查 ModelArts 的 Ascend 运行时。')
npu_count = torch.npu.device_count()
require(npu_count > 0, '没有检测到 NPU。请确认 ModelArts 实例规格和可见设备。')
torch_version = getattr(torch, '__version__', 'unknown')
torch_npu_version = getattr(torch_npu, '__version__', 'unknown')
def major_minor(version: str) -> tuple[int, int] | None:
    try:
        parts = version.split('+', 1)[0].split('.')
        return int(parts[0]), int(parts[1])
    except (IndexError, ValueError):
        return None
if major_minor(torch_version) and major_minor(torch_npu_version):
    require(major_minor(torch_version) == major_minor(torch_npu_version), f'torch 与 torch_npu 版本不匹配：{torch_version} vs {torch_npu_version}。请按同一套 CANN/PyTorch/torch_npu 重新准备 ModelArts 镜像。')
torch.npu.set_device(0)
probe = torch.zeros(1, device='npu:0')
del probe

NOTEBOOK_ID = 'L07-03'
WORK_DIR = Path(os.environ.get('L07_WORK_DIR', str(Path.cwd() / f'{NOTEBOOK_ID}_workspace'))).expanduser()
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get('L07_MODEL_ID', 'Qwen/Qwen3-0.6B')
MODEL_CACHE = Path(os.environ.get('MODELSCOPE_CACHE', str(WORK_DIR / 'modelscope_cache'))).expanduser()
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
MODEL_PATH_OVERRIDE = os.environ.get('L07_MODEL_PATH')
if MODEL_PATH_OVERRIDE:
    MODEL_PATH = Path(MODEL_PATH_OVERRIDE).expanduser()
    require(MODEL_PATH.is_dir(), f'指定的模型目录不存在：{MODEL_PATH}')
else:
    from modelscope import snapshot_download
    MODEL_PATH = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE)))
require((MODEL_PATH / 'config.json').is_file(), f'模型目录缺少 config.json：{MODEL_PATH}')
MODEL_REF = str(MODEL_PATH)

def qwen3_answer(text: str) -> str:
    return '<think>\n\n</think>\n\n' + text

examples = [
    ('请用一句话解释 LoRA 与全参数微调的区别。', 'LoRA 只训练注入的低秩适配器参数，全参数微调会更新模型的全部参数。'),
    ('lora_rank 控制什么？', '它是低秩分支的维度，决定适配器的参数量和表达容量。'),
    ('lora_alpha 控制什么？', '它控制低秩更新的缩放，不等同于 learning_rate。'),
    ('target_modules=all-linear 表示什么？', '它表示向模型中的线性层注入 LoRA 适配器。'),
    ('一个 batch 有 2 条样本，梯度累积 4 步时有效 batch size 是多少？', '单卡且不丢弃样本时，有效 batch size 是 2 乘以 4，也就是 8。'),
    ('SFT 数据中的 assistant 消息有什么作用？', '它提供模型需要学习的目标答案。'),
]
records = [
    {'messages': [
        {'role': 'system', 'content': '你是一个简洁、准确的课程实验助手。'},
        {'role': 'user', 'content': question + ' /no_think'},
        {'role': 'assistant', 'content': qwen3_answer(answer)},
    ]}
    for question, answer in examples
]
TRAIN_REPEATS = 9
records = [record for _ in range(TRAIN_REPEATS) for record in records]
TRAIN_DATA = WORK_DIR / 'train.jsonl'
with TRAIN_DATA.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
loaded_records = [json.loads(line) for line in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
require(len(loaded_records) == len(records), '生成的 JSONL 条数不一致。')
require(all('messages' in item for item in loaded_records), '每条训练数据都必须包含 messages。')
environment_record = {'python': sys.version.split()[0], 'torch': getattr(torch, '__version__', 'unknown'), 'torch_npu': getattr(torch_npu, '__version__', 'unknown'), 'npu_count': npu_count, 'visible_npus': os.environ['ASCEND_RT_VISIBLE_DEVICES'], 'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA)}
(WORK_DIR / 'environment.json').write_text(json.dumps(environment_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

SWIFT_BIN = shutil.which('swift')
require(SWIFT_BIN is not None, '找不到 swift 命令；请确认 ms-swift 已安装且当前 Python 的 bin 目录在 PATH 中。')
VISIBLE_NPUS = os.environ['ASCEND_RT_VISIBLE_DEVICES']
print({'model_id': MODEL_ID, 'model_path': MODEL_REF, 'train_data': str(TRAIN_DATA), 'work_dir': str(WORK_DIR), 'npu_count': npu_count, 'visible_npus': VISIBLE_NPUS})

def run_streaming(command: list[str], log_path: Path, env: dict[str, str]) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('将执行：\n' + shlex.join(command))
    with log_path.open('w', encoding='utf-8') as stream:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            stream.write(line)
        returncode = process.wait()
    require(returncode == 0, f'命令失败（退出码 {returncode}）。请检查：{log_path}')
    print('命令输出已保存到：', log_path)


In [ ]:
# 这些参数是本节观察 LoRA 训练过程的起点。
LORA_RANK = 8
LORA_ALPHA = 32
LEARNING_RATE = 1e-4
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_LENGTH = 512
print('环境、模型和本地 JSONL 数据已准备好。')


In [ ]:
def read_first_record(path: Path) -> dict:
    raw = path.read_text(encoding='utf-8').lstrip()
    require(raw, f'数据文件为空：{path}')
    if raw.startswith('['):
        records = json.loads(raw)
        require(isinstance(records, list) and records and isinstance(records[0], dict), 'JSON 数组的第一项必须是对象。')
        return records[0]
    return json.loads(next(line for line in raw.splitlines() if line.strip()))

first_record = read_first_record(TRAIN_DATA)
require(isinstance(first_record, dict), '训练数据的首条记录必须是对象。')
keys = sorted(first_record.keys())
accepted = {'messages', 'conversations', 'instruction', 'query', 'response'}
require(set(keys) & accepted, f'未识别到常用指令数据字段，当前仅看到：{keys}')
if 'messages' in first_record:
    require(isinstance(first_record['messages'], list) and first_record['messages'], 'messages 必须是非空列表。')
    require({'role', 'content'} <= set(first_record['messages'][0]), 'messages 的首项至少需要 role 与 content 字段。')
print('数据结构检查通过：', keys)


## 2. 训练配置

本次配置把重点放在 LoRA 参数和训练步数：

| 参数 | 本次取值 | 作用 |
| --- | ---: | --- |
| `lora_rank` | 8 | 低秩分支的维度 `r` |
| `lora_alpha` | 32 | 低秩更新的缩放系数 |
| `target_modules` | `all-linear` | 向线性层注入 LoRA |
| `learning_rate` | `1e-4` | 更新适配器参数的步长 |
| `per_device_train_batch_size` | 1 | 每步处理的样本数 |
| `gradient_accumulation_steps` | 1 | 累积梯度的次数 |
| `max_length` | 512 | 输入序列最大 token 数 |
| `max_steps` | 50 | 参数更新次数 |

有效 batch size 的计算式是：`per-device batch size × gradient accumulation steps × data parallel size`。本实验单卡运行，因此有效 batch size 为 1。


### 训练步数和输出

`NUM_TRAIN_EPOCHS=1`，当前数据有 54 条记录；`MAX_STEPS=50` 会在第一个 epoch 内完成训练。`LOGGING_STEPS=1` 让每个 step 都记录 loss，`SAVE_STEPS=10` 每 10 步保存一次 checkpoint。

训练完成后，输出目录中会包含 `notebook_stdout.log`、`logging.jsonl`、`loss_curve.png` 和 LoRA checkpoint。


### 参数之间的关系

`rank` 增大时，`A`、`B` 的参数量增加，适配器能表达更复杂的更新；`alpha` 改变低秩分支的缩放；学习率决定每个 step 的更新幅度。三者作用不同。

如果要做对比实验，先固定模型、数据、序列长度和 batch，再只改变 `rank` 或学习率中的一项。


In [ ]:
# 用完整训练观察 loss 随训练推进的变化。
RUN_NAME = 'L07-03_train'
RUN_DIR = WORK_DIR / RUN_NAME
NUM_TRAIN_EPOCHS = 1
MAX_STEPS = 50
SAVE_STEPS = 10
LOGGING_STEPS = 1
NPROC_PER_NODE = int(os.environ.get('L07_NPROC_PER_NODE', '1'))

train_command = [
    SWIFT_BIN, 'sft',
    '--model', MODEL_REF,
    '--dataset', str(TRAIN_DATA),
    '--torch_dtype', 'bfloat16',
    '--tuner_type', 'lora',
    '--target_modules', 'all-linear',
    '--lora_rank', str(LORA_RANK),
    '--lora_alpha', str(LORA_ALPHA),
    '--loss_scale', 'ignore_empty_think',
    '--num_train_epochs', str(NUM_TRAIN_EPOCHS),
    '--max_steps', str(MAX_STEPS),
    '--per_device_train_batch_size', str(PER_DEVICE_BATCH_SIZE),
    '--gradient_accumulation_steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--learning_rate', str(LEARNING_RATE),
    '--max_length', str(MAX_LENGTH),
    '--logging_steps', str(LOGGING_STEPS),
    '--save_steps', str(SAVE_STEPS),
    '--save_total_limit', '2',
    '--output_dir', str(RUN_DIR),
]
train_env = os.environ.copy()
train_env['ASCEND_RT_VISIBLE_DEVICES'] = VISIBLE_NPUS
train_env['NPROC_PER_NODE'] = str(NPROC_PER_NODE)
train_env['PYTHONUNBUFFERED'] = '1'
print('NPROC_PER_NODE:', NPROC_PER_NODE)
print('train_records:', len(loaded_records), 'max_steps:', MAX_STEPS)
print('训练配置已生成。')


### 日志与图上的点

`LOGGING_STEPS` 决定 loss 的记录间隔，`SAVE_STEPS` 决定 checkpoint 的保存间隔。本实验两者都按固定 step 设置，图上的每个点对应一条训练日志。


## 3. 运行 50-step 训练

运行后观察 `global_step/max_steps` 是否从 `1/50` 推进到 `50/50`，以及每个 step 是否写出 loss。完整输出会写入 `notebook_stdout.log`。


In [ ]:
stdout_path = RUN_DIR / 'notebook_stdout.log'
run_streaming(train_command, stdout_path, train_env)


### 训练过程中观察

看三项：step 是否推进，loss 是否持续写入，训练进程是否正常结束。raw loss 可能上下波动，先看完整曲线，再结合学习率、batch 和数据顺序解释。


## 4. 从日志绘制 loss

代码从 `logging.jsonl` 读取逐 step 的 `loss`，用 step 作为横轴。橙色线是 5-step moving average，用来观察整体趋势；蓝色 raw loss 保留每个 step 的原始波动。


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

logging_files = sorted(RUN_DIR.rglob('logging.jsonl'), key=lambda path: path.stat().st_mtime)
require(logging_files, f'没有找到 logging.jsonl。请检查 {stdout_path}')
logging_file = logging_files[-1]
run_dir = logging_file.parent
step_loss_records = []
fallback_records = []
for raw in logging_file.read_text(encoding='utf-8').splitlines():
    try:
        record = json.loads(raw)
    except json.JSONDecodeError:
        continue
    loss = record.get('loss')
    target_records = step_loss_records
    if not isinstance(loss, (int, float)) or not math.isfinite(float(loss)):
        loss = record.get('train_loss')
        target_records = fallback_records
    if isinstance(loss, (int, float)) and math.isfinite(float(loss)):
        step = record.get('global_step', record.get('current_steps'))
        if not isinstance(step, (int, float)):
            for key in ('global_step/max_steps', 'iteration'):
                value = record.get(key)
                if isinstance(value, str) and '/' in value:
                    left = value.split('/', 1)[0].strip()
                    if left.isdigit():
                        step = int(left)
                        break
        target_records.append({'step': step if isinstance(step, (int, float)) else len(target_records) + 1, 'loss': float(loss), 'raw': record})
records = step_loss_records or fallback_records
deduplicated = {}
for item in records:
    deduplicated.setdefault(item['step'], item)
records = [deduplicated[step] for step in sorted(deduplicated)]
require(records, f'未从 {logging_file} 解析到 loss / train_loss 字段。请记录实际字段名，不要手填曲线。')
checkpoints = [path for path in run_dir.glob('checkpoint-*') if path.is_dir()]
require(checkpoints, f'没有在 {run_dir} 找到 checkpoint。')

steps = [item['step'] for item in records]
losses = [item['loss'] for item in records]
require(max(steps) >= MAX_STEPS, f'有效 loss 日志最高 step 为 {max(steps)}，低于要求的 {MAX_STEPS}。')
plt.figure(figsize=(8, 4.5))
plt.plot(steps, losses, marker='o', linewidth=1.2, markersize=3, label='raw loss')
if len(losses) >= 5:
    window = min(5, len(losses))
    smooth = [sum(losses[max(0, i-window+1):i+1]) / len(losses[max(0, i-window+1):i+1]) for i in range(len(losses))]
    plt.plot(steps, smooth, linewidth=2, label=f'moving average ({window})')
plt.xlabel('training step')
plt.ylabel('loss')
plt.title('L07-03 training loss (this run only)')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
figure_path = RUN_DIR / 'loss_curve.png'
plt.savefig(figure_path, dpi=150)
plt.show()
print('loss points:', len(records))
print('raw log:', logging_file)
print('checkpoint:', checkpoints[-1])
print('figure:', figure_path)


## 5. 读懂 loss 曲线

观察曲线时回答下面的问题：

1. raw loss 的最高点和最低点分别出现在什么 step？
2. moving average 从训练开始到结束是上升、下降还是基本不变？
3. 后半段的波动是否比前半段小？这可能和适配器逐步拟合当前样本有关。
4. 如果把 `lora_rank` 改成 16，你预计曲线和 checkpoint 会有什么变化？

移动平均只用于看趋势，不能代替 raw loss。


## 6. 可选：观察训练内存

如果训练日志包含 `memory(GiB)` 字段，下面的代码会把它画出来，帮助观察训练过程中的内存记录。


In [ ]:
memory_records = []
for item in records:
    value = item['raw'].get('memory(GiB)')
    if isinstance(value, (int, float)):
        memory_records.append((item['step'], float(value)))
if memory_records:
    x, y = zip(*memory_records)
    plt.figure(figsize=(8, 3.5))
    plt.plot(x, y, marker='o', linewidth=1.2)
    plt.xlabel('training step')
    plt.ylabel('memory (GiB)')
    plt.title('Memory records emitted by this run')
    plt.grid(alpha=0.25)
    plt.tight_layout()
    memory_path = RUN_DIR / 'memory_from_logging_jsonl.png'
    plt.savefig(memory_path, dpi=150)
    plt.show()
    print('memory figure:', memory_path)
else:
    print('当前 logging.jsonl 没有 memory(GiB) 字段。')


## 内存曲线怎么读

内存记录来自训练日志，横轴仍是 step。结合 `max_length`、batch size 和精度，可以思考哪些设置会影响 LoRA 训练的内存占用。


## 7. 训练后练习

1. 写出 LoRA 更新公式，并标出本实验中的 `r=8` 和 `alpha=32`。
2. 根据曲线描述训练前 10 step 和后 10 step 的差别。
3. 为什么 raw loss 会波动，而 moving average 可能持续下降？
4. 如果只训练 `q_proj`、`v_proj`，与 `all-linear` 相比会改变什么？
5. 选择一个参数（`rank`、`alpha` 或学习率）做下一次对比，写出你的预期。


## 提交前检查

- [ ] 写出基础模型、LoRA 参数、数据条数和训练步数。
- [ ] 保存 `logging.jsonl`、`notebook_stdout.log`、`loss_curve.png` 和 checkpoint 路径。
- [ ] 在报告中解释 raw loss 与 moving average 的区别。
- [ ] 画出或记录下一次调参实验的预期。
